# 04 — Pilot: pick the best proposed-model variant
Fold 0, seed 42, validation set only (test set untouched). Candidates: {ConvNeXt-Tiny, EfficientNetV2-S} × {ECA, CBAM}.

In [1]:
import sys, json
from pathlib import Path
import torch
import pandas as pd

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
SPLITS_DIR = PROJECT_ROOT / "splits"
CACHE_DIR  = PROJECT_ROOT / "cache_384"
SRC_DIR    = PROJECT_ROOT / "src"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR.mkdir(exist_ok=True); RESULTS_DIR.mkdir(exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Device: cuda | NVIDIA GeForce RTX 3050


In [2]:

from pneumo_data import *
from pneumo_models import *
from pneumo_train import *

PILOT_FOLD, PILOT_SEED = 0, 42
PILOT_EPOCHS, PILOT_PATIENCE = 15, 5
BATCH, WORKERS = 16, 4
PILOT_DIR = RESULTS_DIR / "pilot"
CANDIDATES = [f"proposed:{b}:{a}" for b in ("convnext_tiny", "efficientnet_v2_s") for a in ("eca", "cbam")]
print(CANDIDATES)


['proposed:convnext_tiny:eca', 'proposed:convnext_tiny:cbam', 'proposed:efficientnet_v2_s:eca', 'proposed:efficientnet_v2_s:cbam']


In [ ]:

# ============================================================
# Train every candidate (resumable: finished runs are skipped)
# ============================================================
rows = []
for spec in CANDIDATES:
    run_dir = PILOT_DIR / spec.replace(":", "_")
    if (run_dir / "summary.json").exists():
        s = json.loads((run_dir / "summary.json").read_text())
        print("Skip (done):", spec)
    else:
        print("\n=== ", spec, " ===")
        set_seed(PILOT_SEED)
        train_loader, val_loader, _, cw, _ = make_loaders(
            PILOT_FOLD, SPLITS_DIR, CACHE_DIR, batch_size=BATCH, num_workers=WORKERS, seed=PILOT_SEED)
        model = build_model(spec, pretrained=True)
        model, s = train_run(model, train_loader, val_loader, cw, run_dir, device,
                             dict(epochs=PILOT_EPOCHS, patience=PILOT_PATIENCE))
        del model; torch.cuda.empty_cache()
    rows.append({"model": spec, "best_val_acc": s["best_val_acc"], "best_val_macro_f1": s["best_val_macro_f1"],
                 "best_epoch": s["best_epoch"], "params_M": s["params"] / 1e6, "minutes": s["minutes"]})
res = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False).reset_index(drop=True)
res.to_csv(PILOT_DIR / "pilot_results.csv", index=False)
res



===  proposed:convnext_tiny:eca  ===


In [ ]:

# ============================================================
# Select by validation accuracy (near-tie within 0.5 pt ≈ 6 val images -> fewer params)
# ============================================================
top = res["best_val_acc"].max()
near = res[res["best_val_acc"] >= top - 0.005].sort_values("params_M")
sel = near.iloc[0]
_, backbone, attn = sel["model"].split(":")
selected = {"spec": sel["model"], "backbone": backbone, "attn": attn,
            "val_acc": float(sel["best_val_acc"]), "params_M": float(sel["params_M"])}
(PROJECT_ROOT / "selected_model.json").write_text(json.dumps(selected, indent=2))
print("Selected:", json.dumps(selected, indent=2))
